# Setup check · Physics-informed machine learning for civil engineers

Run this once before the course. It takes about 30 seconds.

**In Colab:** choose **Runtime → Run all** (or press the ▶ button on the code cell below).

It checks that PyTorch, NumPy and Matplotlib are available, that automatic differentiation works up to the fourth derivative (the beam equation needs it), and that a tiny physics-informed network trains. The last line should say **You are all set up**.

No GPU is needed: everything in the course runs on the free Colab CPU.

In [ ]:
# Setup check for the PIML course. Run this cell (or Runtime -> Run all).
import sys, time, platform

results = []
def report(name, ok, detail=""):
    results.append(ok)
    print(f"{'✓' if ok else '✗'}  {name:<38} {detail}")

print("Checking your setup...\n")

# 1. Python
v = sys.version_info
report("Python 3.9 or newer", v >= (3, 9), f"{v.major}.{v.minor}.{v.micro}")

# 2. Libraries
torch = np = plt = None
try:
    import torch
    major, minor = (int(p) for p in torch.__version__.split("+")[0].split(".")[:2])
    report("PyTorch 2.0 or newer", (major, minor) >= (2, 0), torch.__version__)
except Exception as e:
    report("PyTorch 2.0 or newer", False, f"not found ({e.__class__.__name__})")
try:
    import numpy as np
    report("NumPy", True, np.__version__)
except Exception as e:
    report("NumPy", False, f"not found ({e.__class__.__name__})")
try:
    import matplotlib
    import matplotlib.pyplot as plt
    report("Matplotlib", True, matplotlib.__version__)
except Exception as e:
    report("Matplotlib", False, f"not found ({e.__class__.__name__})")

if torch is not None:
    torch.set_default_dtype(torch.float64)
    dev = "GPU (" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "CPU"
    print(f"   device: {dev}  (the course runs on the CPU)")

    # 3. Automatic differentiation up to the 4th derivative (the beam equation needs it)
    try:
        x = torch.linspace(0.1, 1.0, 5, requires_grad=True)
        y = x**4
        d = y
        for _ in range(4):
            d = torch.autograd.grad(d, x, torch.ones_like(d), create_graph=True)[0]
        ok = torch.allclose(d, torch.full_like(d, 24.0))
        report("autograd 4th derivative (d⁴x⁴/dx⁴ = 24)", ok, f"got {d[0].item():.4f}")
    except Exception as e:
        report("autograd 4th derivative (d⁴x⁴/dx⁴ = 24)", False, str(e)[:60])

    # 4. A tiny PINN: u'' = -1 on [0, 1], u(0) = u(1) = 0, exact u = x(1 - x)/2
    try:
        torch.manual_seed(0)
        net = torch.nn.Sequential(torch.nn.Linear(1, 20), torch.nn.Tanh(),
                                  torch.nn.Linear(20, 20), torch.nn.Tanh(),
                                  torch.nn.Linear(20, 1))
        opt = torch.optim.Adam(net.parameters(), lr=1e-2)
        xc = torch.linspace(0, 1, 50).reshape(-1, 1).requires_grad_(True)
        xb = torch.tensor([[0.0], [1.0]])
        def loss_fn():
            u = net(xc)
            du = torch.autograd.grad(u, xc, torch.ones_like(u), create_graph=True)[0]
            d2u = torch.autograd.grad(du, xc, torch.ones_like(du), create_graph=True)[0]
            return ((d2u + 1)**2).mean() + (net(xb)**2).mean()
        t0 = time.time()
        first = loss_fn().item()
        for _ in range(1000):
            opt.zero_grad(); loss = loss_fn(); loss.backward(); opt.step()
        last = loss.item()
        with torch.no_grad():
            xt = torch.linspace(0, 1, 101).reshape(-1, 1)
            err = ((net(xt) - xt*(1 - xt)/2).abs().max() / 0.125).item()
        report("tiny PINN trains (loss falls 100x)", last < first / 100,
               f"loss {first:.2e} -> {last:.2e} in {time.time()-t0:.1f} s, error {100*err:.1f}%")
    except Exception as e:
        report("tiny PINN trains (loss falls 100x)", False, str(e)[:60])

# 5. Plotting
if plt is not None and np is not None:
    try:
        xs = np.linspace(0, 1, 50)
        fig, ax = plt.subplots(figsize=(4, 2.2))
        ax.plot(xs, xs*(1 - xs)/2); ax.set_title("Plotting works"); ax.set_xlabel("x")
        plt.tight_layout(); plt.show()
        report("Matplotlib plots", True)
    except Exception as e:
        report("Matplotlib plots", False, str(e)[:60])

print()
if all(results):
    print("=" * 46)
    print("  ✅  You are all set up. See you in the course!")
    print("=" * 46)
else:
    print("❌  Something is missing (see the ✗ lines above).")
    print("    In Colab: Runtime -> Disconnect and delete runtime, then run again.")
    print("    On your own laptop: pip install torch numpy matplotlib")
    print("    Still stuck? Send a screenshot of this output to the instructor.")